In [ ]:
import pandas as pd
import networkx as nx

# Load the dataset
df = pd.read_csv("/content/combined_coauthors.csv")

# Initialize a graph
G = nx.Graph()

# Dictionary to store author class (Awardee/Nonawardee)
author_classes = {}

# Iterate through each row and create edges
for _, row in df.iterrows():
    author = row["Author"]
    coauthors = str(row["Coauthors"]).split(", ")  # Split coauthors by comma
    author_classes[author] = row["Class"]  # Store class

    for coauthor in coauthors:
        if coauthor and coauthor != author:  # Avoid self-loops
            G.add_edge(author, coauthor)
            author_classes[coauthor] = df.loc[df["Author"] == coauthor, "Class"].values[0] if coauthor in df["Author"].values else "Nonawardee"

In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
from tqdm import tqdm

def remove_cycles_efficiently(G):
    """
    Efficiently remove cycles in a large directed graph to convert it into a DAG.
    Instead of storing all cycles, we iteratively remove edges until a DAG is achieved.

    Parameters:
    - G: Directed graph (DiGraph)

    Returns:
    - DAG (Directed Acyclic Graph)
    """
    cycles_removed = 0
    try:
        while True:
            cycle = nx.find_cycle(G, orientation="original")  # Find one cycle
            if not cycle:
                break  # No more cycles found

            edge_to_remove = cycle[0]  # Remove only the first edge in the cycle
            G.remove_edge(*edge_to_remove[:2])
            cycles_removed += 1
    except nx.NetworkXNoCycle:
        pass  # No cycles found

    print(f"[INFO] Removed {cycles_removed} cycles to convert into a DAG.")
    return G

def second_order_centrality(G):
    """
    Compute Second Order Centrality for all nodes in a large graph.

    Parameters:
    - G: NetworkX graph

    Returns:
    - Dictionary of node second-order centrality scores
    """
    print("\n[INFO] Computing Second Order Centrality...")
    soc_scores = {}

    for node in tqdm(G.nodes(), total=len(G.nodes())):
        try:
            # Compute shortest path lengths from this node
            sp_lengths = nx.single_source_shortest_path_length(G, node)
            distances = list(sp_lengths.values())

            if len(distances) > 1:
                soc_scores[node] = np.std(distances)  # Standard deviation of shortest paths
            else:
                soc_scores[node] = 0.0  # Isolated nodes get 0 SOC

        except Exception as e:
            print(f"[ERROR] Could not compute SOC for {node}: {e}")
            soc_scores[node] = 0.0

    return soc_scores

# Initialize graph
G = nx.Graph()
author_classes = {}

print("[INFO] Building Graph from dataset...")

# Read CSV in chunks to avoid memory overload
for chunk in tqdm(pd.read_csv("/content/combined_coauthors.csv", chunksize=10000), desc="Processing Chunks"):
    chunk.fillna("", inplace=True)

    # Bulk process nodes
    authors = set(chunk["Author"].astype(str).str.strip())
    G.add_nodes_from(authors)

    # Construct edges
    edges = []
    for _, row in chunk.iterrows():
        author = str(row["Author"]).strip()
        coauthors = str(row["Coauthors"]).split(",")

        for coauthor in coauthors:
            coauthor = coauthor.strip()
            if coauthor and coauthor != author:
                edges.append((coauthor, author))

    G.add_edges_from(edges)  # Add edges in bulk to reduce memory usage

# Remove cycles efficiently (only if using a directed graph)
if G.is_directed():
    G = remove_cycles_efficiently(G)

# Compute Second Order Centrality
soc_scores = second_order_centrality(G)

# Save results in chunks to avoid RAM overflow
output_file = "author_centralities_SecondOrder.csv"
print("[INFO] Saving results in chunks to avoid RAM overload...")
chunk_size = 10000
soc_list = list(soc_scores.items())

for i in range(0, len(soc_list), chunk_size):
    batch = soc_list[i:i + chunk_size]
    batch_df = pd.DataFrame(batch, columns=["Author", "Second_Order_Centrality"])
    batch_df["Class"] = batch_df["Author"].map(lambda x: author_classes.get(x, "Nonawardee"))
    batch_df.to_csv(output_file, mode='a', index=False, header=(i == 0))  # Append to file

print("\n[INFO] Second Order Centrality calculated and saved as 'author_centralities_SecondOrder.csv'.")


[INFO] Building Graph from dataset...


Processing Chunks: 3it [00:02,  1.36it/s]



[INFO] Computing Second Order Centrality...


100%|██████████| 52536/52536 [59:02<00:00, 14.83it/s]


[INFO] Saving results in chunks to avoid RAM overload...

[INFO] Second Order Centrality calculated and saved as 'author_centralities_SecondOrder.csv'.


In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
from tqdm import tqdm

def dispersion_centrality(G):
    """
    Compute Dispersion Centrality for all nodes in a large graph.

    Parameters:
    - G: NetworkX graph

    Returns:
    - Dictionary of node dispersion centrality scores
    """
    print("\n[INFO] Computing Dispersion Centrality...")
    dispersion_scores = {}

    for node in tqdm(G.nodes(), total=len(G.nodes())):
        try:
            dispersion_dict = nx.dispersion(G, u=node)  # Returns {neighbor: score}
            if dispersion_dict:
                dispersion_scores[node] = np.mean(list(dispersion_dict.values()))  # Average dispersion score
            else:
                dispersion_scores[node] = 0.0  # No valid dispersion score
        except Exception as e:
            print(f"[ERROR] Could not compute Dispersion for {node}: {e}")
            dispersion_scores[node] = 0.0

    return dispersion_scores

# Initialize graph
G = nx.Graph()
author_classes = {}

print("[INFO] Building Graph from dataset...")

# Read CSV in chunks to avoid memory overload
for chunk in tqdm(pd.read_csv("/content/combined_coauthors.csv", chunksize=10000), desc="Processing Chunks"):
    chunk.fillna("", inplace=True)

    # Bulk process nodes
    authors = set(chunk["Author"].astype(str).str.strip())
    G.add_nodes_from(authors)

    # Construct edges
    edges = []
    for _, row in chunk.iterrows():
        author = str(row["Author"]).strip()
        coauthors = str(row["Coauthors"]).split(",")

        for coauthor in coauthors:
            coauthor = coauthor.strip()
            if coauthor and coauthor != author:
                edges.append((coauthor, author))

    G.add_edges_from(edges)  # Add edges in bulk to reduce memory usage

# Compute Dispersion Centrality
dispersion_scores = dispersion_centrality(G)

# Convert Dispersion Scores to max normalized values
max_value = max(dispersion_scores.values()) if dispersion_scores else 1
dispersion_scores = {node: score / max_value for node, score in dispersion_scores.items()}

# Save results in chunks to avoid RAM overflow
output_file = "author_centralities_Dispersion.csv"
print("[INFO] Saving results in chunks to avoid RAM overload...")
chunk_size = 10000
dispersion_list = list(dispersion_scores.items())

for i in range(0, len(dispersion_list), chunk_size):
    batch = dispersion_list[i:i + chunk_size]
    batch_df = pd.DataFrame(batch, columns=["Author", "Dispersion_Centrality"])
    batch_df["Class"] = batch_df["Author"].map(lambda x: author_classes.get(x, "Nonawardee"))
    batch_df.to_csv(output_file, mode='a', index=False, header=(i == 0))  # Append to file

print("\n[INFO] Dispersion Centrality calculated and saved as 'author_centralities_Dispersion.csv'.")


[INFO] Building Graph from dataset...


Processing Chunks: 3it [00:02,  1.44it/s]



[INFO] Computing Dispersion Centrality...


100%|██████████| 52536/52536 [02:40<00:00, 328.25it/s]  


[INFO] Saving results in chunks to avoid RAM overload...

[INFO] Dispersion Centrality calculated and saved as 'author_centralities_Dispersion.csv'.


In [ ]:
print(centrality_df.head())  # Check if data is correctly populated
print("Nodes in Graph:", G.number_of_nodes())

   Author  Information_Centrality       Class
0       0            1.547546e+16  Nonawardee
1       1            5.258479e+15  Nonawardee
2       2            4.803840e+15  Nonawardee
3       3            6.181885e+15  Nonawardee
4       4           -3.798753e+15  Nonawardee
Nodes in Graph: 100


In [ ]:
import networkx as nx
import pandas as pd

# Compute Clustering Coefficient for all nodes
clustering_centrality = nx.clustering(G)

# Compute Degree Centrality for all nodes (needed for Cluster Rank Centrality)
degree_centrality = nx.degree_centrality(G)

# Compute Cluster Rank Centrality (Clustering * Degree)
cluster_rank_centrality = {node: clustering_centrality[node] * degree_centrality[node] for node in G.nodes()}

# Convert to DataFrame with Author, Awardee Status, and Centrality


centrality_df = pd.DataFrame({
    "Author": list(cluster_rank_centrality.keys()),  # Node names
    "Cluster_Rank_Centrality": list(cluster_rank_centrality.values()),  # Cluster Rank values
    "Class": [author_classes.get(node, "Nonawardee") for node in cluster_rank_centrality.keys()],  # Awardee status

})

# Save to CSV
centrality_df.to_csv("author_centralities_ClusterRank.csv", index=False)

print("Cluster Rank Centrality calculated and saved as 'author_centralities_ClusterRank.csv'.")


NameError: name 'G' is not defined

In [ ]:
import networkx as nx
import pandas as pd

# Define the decay factor (adjustable based on needs)
delta = 0.85

# Compute shortest path lengths for all pairs
shortest_paths = dict(nx.all_pairs_shortest_path_length(G))

# Compute Decay Centrality for all nodes
decay_centrality = {
    node: sum(delta ** shortest_paths[node].get(neighbor, float('inf')) for neighbor in G.nodes())
    for node in G.nodes()
}

# Convert to DataFrame with Author, Awardee Status, and Centrality
centrality_df = pd.DataFrame({
    "Author": list(decay_centrality.keys()),  # Node names
    "Decay_Centrality": list(decay_centrality.values()),  # Decay Centrality values
    "Class": [author_classes.get(node, "Nonawardee") for node in decay_centrality.keys()],  # Awardee status
})

# Save to CSV
centrality_df.to_csv("author_centralities_Decay.csv", index=False)

print("Decay Centrality calculated and saved as 'author_centralities_Decay.csv'.")


In [ ]:
import pickle
import pandas as pd
import networkx as nx
# Save the graph and author_classes
with open("/content/graph.pkl", "wb") as f:
    pickle.dump(G, f)

with open("/content/author_classes.pkl", "wb") as f:
    pickle.dump(author_classes, f)


In [ ]:
import pickle
import pandas as pd
import networkx as nx
G = nx.Graph()
author_classes = {}

with open("/content/graph.pkl", "rb") as f:
    G = pickle.load(f)

with open("/content/author_classes.pkl", "rb") as f:
    author_classes = pickle.load(f)


In [ ]:
import networkx as nx
import pandas as pd

# Compute PageRank Centrality for all nodes
pagerank_centrality = nx.pagerank(G, alpha=0.85)  # Alpha=0.85 is the default damping factor

# Create DataFrame with Author, PageRank Centrality, and Class Label
centrality_df = pd.DataFrame({
    "Author": list(pagerank_centrality.keys()),
    "PageRank_Centrality": list(pagerank_centrality.values()),
    "Class": [author_classes.get(node, "Nonawardee") for node in pagerank_centrality.keys()]
})

# Save to CSV
centrality_df.to_csv("author_centralities_PageRank.csv", index=False)

print("PageRank Centrality calculated and saved as 'author_centralities_PageRank.csv'.")


PageRank Centrality calculated and saved as 'author_centralities_PageRank.csv'.


In [ ]:
import networkx as nx
import pandas as pd

# Compute Local Reaching Centrality for all nodes
local_reaching_centrality = {node: nx.local_reaching_centrality(G, node) for node in G.nodes()}

# Create DataFrame with Author, Local Reaching Centrality, and Class Label
centrality_df = pd.DataFrame({
    "Author": list(local_reaching_centrality.keys()),
    "Local_Reaching_Centrality": list(local_reaching_centrality.values()),
    "Class": [author_classes.get(node, "Nonawardee") for node in local_reaching_centrality.keys()]
})

# Save to CSV
centrality_df.to_csv("author_centralities_LocalReaching.csv", index=False)

print("Local Reaching Centrality calculated and saved as 'author_centralities_LocalReaching.csv'.")


In [ ]:
import networkx as nx
import pandas as pd

# Compute Local Reaching Centrality for all nodes (required for GRC calculation)
local_reaching_centrality = {node: nx.local_reaching_centrality(G, node) for node in G.nodes()}

# Compute Global Reaching Centrality (GRC)
min_lrc = min(local_reaching_centrality.values())  # Minimum Local Reaching Centrality
global_reaching_centrality = sum(lrc - min_lrc for lrc in local_reaching_centrality.values())

# Create DataFrame with Author, Global Reaching Centrality, and Class Label
centrality_df = pd.DataFrame({
    "Author": list(local_reaching_centrality.keys()),
    "Global_Reaching_Centrality": [global_reaching_centrality] * len(local_reaching_centrality),  # Same GRC for all nodes
    "Class": [author_classes.get(node, "Nonawardee") for node in local_reaching_centrality.keys()]
})

# Save to CSV
centrality_df.to_csv("author_centralities_GlobalReaching.csv", index=False)

print("Global Reaching Centrality calculated and saved as 'author_centralities_GlobalReaching.csv'.")


In [ ]:
import networkx as nx
import pandas as pd

# Compute VoteRank centrality (returns a ranked list of nodes)
voterank = nx.voterank(G, len(G.nodes()))  # Limit to top 100 for efficiency

# Create a dictionary mapping nodes to their rank (lower rank = higher importance)
voterank_dict = {node: rank for rank, node in enumerate(voterank)}

# Create DataFrame with Author, VoteRank Centrality, and Class
centrality_df = pd.DataFrame({
    "Author": list(G.nodes()),
    "VoteRank_Centrality": [voterank_dict.get(node, float('inf')) for node in G.nodes()],  # Unranked nodes get 'inf'
    "Class": [author_classes.get(node, "Nonawardee") for node in G.nodes()]
})

# Save updated results
centrality_df.to_csv("author_centralities_VoteRank.csv", index=False)

print("VoteRank centrality calculated and saved as 'author_centralities_VoteRank.csv'.")


VoteRank centrality calculated and saved as 'author_centralities_VoteRank.csv'.


In [ ]:
import networkx as nx
import pandas as pd

# Compute Harmonic Centrality for all nodes using NetworkX's built-in function
harmonic_centrality = nx.harmonic_centrality(G)

# Create DataFrame with Author, Harmonic Centrality, and Class Label
centrality_df = pd.DataFrame({
    "Author": list(harmonic_centrality.keys()),
    "Harmonic_Centrality": list(harmonic_centrality.values()),
    "Class": [author_classes.get(node, "Nonawardee") for node in harmonic_centrality.keys()]
})

# Save to CSV
centrality_df.to_csv("author_centralities_Harmonic.csv", index=False)

print("Harmonic Centrality calculated and saved as 'author_centralities_Harmonic.csv'.")


Harmonic Centrality calculated and saved as 'author_centralities_Harmonic.csv'.


In [ ]:
import networkx as nx
import pandas as pd
import numpy as np
import pickle
from scipy.sparse import csgraph
from scipy.sparse.linalg import cg  # Conjugate Gradient for efficiency
import os

# Load the dataset and construct the graph as before
# Assume 'G' and 'author_classes' are already available

# Convert Graph Laplacian to a Sparse Matrix (Memory Efficient)
L = nx.laplacian_matrix(G).tocsc()

# Create a mapping between node labels and matrix indices
node_to_index = {node: i for i, node in enumerate(G.nodes())}
index_to_node = {i: node for node, i in node_to_index.items()}

# Checkpoint file to resume progress
checkpoint_file = "information_centrality_checkpoint.pkl"
progress_save_interval = 500  # Save progress every 500 nodes

# Load checkpoint if exists
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, "rb") as f:
        information_centrality, last_processed_idx = pickle.load(f)
    print(f"Resuming from checkpoint: {last_processed_idx} nodes processed.")
else:
    information_centrality = {}
    last_processed_idx = 0

# Function to approximate information centrality
def approximate_information_centrality(L, node_to_index, last_processed_idx, information_centrality):
    n = L.shape[0]

    for idx, (node, i) in enumerate(node_to_index.items()):
        if idx < last_processed_idx:
            continue  # Skip already processed nodes

        # Solve L * x = b using Conjugate Gradient
        b = np.zeros(n)
        b[i] = 1
        x, _ = cg(L, b, tol=1e-5)  # Solve Lx = b

        ic_value = 1 / np.sum(x) if np.sum(x) != 0 else 0
        information_centrality[node] = ic_value

        # Save progress every `progress_save_interval` nodes
        if idx % progress_save_interval == 0:
            with open(checkpoint_file, "wb") as f:
                pickle.dump((information_centrality, idx), f)
            print(f"Checkpoint saved at {idx} nodes...")

    return information_centrality

# Compute Information Centrality with checkpointing
information_centrality = approximate_information_centrality(L, node_to_index, last_processed_idx, information_centrality)

# Create DataFrame with Author, Information Centrality, and Class Label
centrality_df = pd.DataFrame({
    "Author": list(information_centrality.keys()),
    "Information_Centrality": list(information_centrality.values()),
    "Class": [author_classes.get(node, "Nonawardee") for node in information_centrality.keys()]
})

# Save final results
centrality_df.to_csv("author_centralities_Information.csv", index=False)

# Delete checkpoint since computation is complete
if os.path.exists(checkpoint_file):
    os.remove(checkpoint_file)

print("Approximate Information Centrality calculated and saved as 'author_centralities_Information.csv'.")


<ipython-input-4-643fe055f1cd>:43: DeprecationWarning: 'scipy.sparse.linalg.cg' keyword argument `tol` is deprecated in favor of `rtol` and will be removed in SciPy v1.14.0. Until then, if set, it will override `rtol`.
  x, _ = cg(L, b, tol=1e-5)  # Solve Lx = b


Checkpoint saved at 0 nodes...


KeyboardInterrupt: 

In [ ]:
import networkx as nx
import pandas as pd
import pickle  # To save/load progress
import os

# Define the save file
save_file = "eccentricity_progress.pkl"

# Load saved progress if exists
if os.path.exists(save_file):
    with open(save_file, "rb") as f:
        eccentricity_centrality = pickle.load(f)
    print("Resuming from saved progress...")
else:
    eccentricity_centrality = {}

# Compute Eccentricity Centrality (resume if interrupted)
for node in G.nodes():
    if node not in eccentricity_centrality:  # Skip already computed nodes
        try:
            lengths = nx.single_source_shortest_path_length(G, node)
            eccentricity_centrality[node] = max(lengths.values())  # Max shortest path distance

            # Save progress every 500 nodes
            if len(eccentricity_centrality) % 500 == 0:
                with open(save_file, "wb") as f:
                    pickle.dump(eccentricity_centrality, f)
                print(f"Saved progress at {len(eccentricity_centrality)} nodes...")
        except Exception as e:
            print(f"Error processing node {node}: {e}")

# Remove progress file after completion
if os.path.exists(save_file):
    os.remove(save_file)

# Create DataFrame with Author, Eccentricity Centrality, and Class Label
centrality_df = pd.DataFrame({
    "Author": list(eccentricity_centrality.keys()),
    "Eccentricity_Centrality": list(eccentricity_centrality.values()),
    "Class": [author_classes.get(node, "Nonawardee") for node in eccentricity_centrality.keys()]
})

# Save to CSV
centrality_df.to_csv("author_centralities_Eccentricity.csv", index=False)

print("Eccentricity Centrality calculated and saved as 'author_centralities_Eccentricity.csv'.")


Saved progress at 500 nodes...
Saved progress at 1000 nodes...
Saved progress at 1500 nodes...
Saved progress at 2000 nodes...
Saved progress at 2500 nodes...
Saved progress at 3000 nodes...
Saved progress at 3500 nodes...
Saved progress at 4000 nodes...
Saved progress at 4500 nodes...
Saved progress at 5000 nodes...
Saved progress at 5500 nodes...
Saved progress at 6000 nodes...
Saved progress at 6500 nodes...
Saved progress at 7000 nodes...
Saved progress at 7500 nodes...
Saved progress at 8000 nodes...
Saved progress at 8500 nodes...
Saved progress at 9000 nodes...
Saved progress at 9500 nodes...
Saved progress at 10000 nodes...
Saved progress at 10500 nodes...
Saved progress at 11000 nodes...
Saved progress at 11500 nodes...
Saved progress at 12000 nodes...
Saved progress at 12500 nodes...
Saved progress at 13000 nodes...
Saved progress at 13500 nodes...
Saved progress at 14000 nodes...
Saved progress at 14500 nodes...
Saved progress at 15000 nodes...
Saved progress at 15500 nodes.

In [ ]:
import pandas as pd
import networkx as nx
import numpy as np

def decay_centrality(G, decay_factor=0.5):
    """
    Compute Decay Centrality for all nodes in the graph G.

    Parameters:
    - G: NetworkX graph
    - decay_factor: The decay rate applied at each step away from the node

    Returns:
    - Dictionary of node decay centrality scores
    """
    centrality = {}
    for node in G.nodes():
        shortest_paths = nx.single_source_shortest_path_length(G, node)
        centrality[node] = sum(decay_factor ** dist for dist in shortest_paths.values())

    return centrality

# Load dataset
df = pd.read_csv("/content/combined_coauthors.csv").fillna("")

# Create author-class mapping beforehand
author_classes = dict(zip(df["Author"].astype(str).str.strip(), df["Class"]))

# Initialize graph
G = nx.Graph()

# Bulk add authors as nodes
authors = set(df["Author"].astype(str).str.strip())
G.add_nodes_from(authors)

# Construct edges efficiently
edges = set()
for _, row in df.iterrows():
    author = str(row["Author"]).strip()
    coauthors = map(str.strip, row["Coauthors"].split(","))

    for coauthor in coauthors:
        if coauthor and coauthor != author:
            edges.add((author, coauthor))

G.add_edges_from(edges)  # Bulk add edges

# Compute Decay Centrality
decay_scores = decay_centrality(G, decay_factor=0.5)

# Convert results to DataFrame
centrality_df = pd.DataFrame({
    "Author": decay_scores.keys(),
    "Decay_Centrality": decay_scores.values(),
    "Class": [author_classes.get(node, "Nonawardee") for node in decay_scores.keys()]
})

# Save results
centrality_df.to_csv("author_centralities_Decay.csv", index=False)
print("Decay Centrality calculated and saved.")


Decay Centrality calculated and saved.


In [ ]:
import pandas as pd
import networkx as nx

def radiality_centrality(G):
    """
    Compute Radiality Centrality for all nodes in the graph G using an efficient approach.

    Parameters:
    - G: NetworkX graph

    Returns:
    - Dictionary of node radiality centrality scores
    """
    n = len(G.nodes)
    if n <= 1:
        return {node: 0 for node in G.nodes}  # Handle small graphs safely

    # Compute Graph Diameter (Maximum shortest path)
    max_diameter = 0
    for node in G.nodes():
        shortest_paths = nx.single_source_shortest_path_length(G, node)
        if shortest_paths:  # Ensure we have reachable nodes
            max_diameter = max(max_diameter, max(shortest_paths.values()))

    centrality = {}
    for node in G.nodes():
        shortest_paths = nx.single_source_shortest_path_length(G, node)
        total_reachability = sum((max_diameter + 1 - dist) for dist in shortest_paths.values())

        centrality[node] = total_reachability / ((n - 1) * (max_diameter + 1))

    return centrality

# Load dataset
df = pd.read_csv("/content/combined_coauthors.csv").fillna("")

# Create author-class mapping beforehand
author_classes = dict(zip(df["Author"].astype(str).str.strip(), df["Class"]))

# Initialize graph
G = nx.Graph()

# Bulk add authors as nodes
authors = set(df["Author"].astype(str).str.strip())
G.add_nodes_from(authors)

# Construct edges efficiently
edges = set()
for _, row in df.iterrows():
    author = str(row["Author"]).strip()
    coauthors = map(str.strip, row["Coauthors"].split(","))

    for coauthor in coauthors:
        if coauthor and coauthor != author:
            edges.add((author, coauthor))

G.add_edges_from(edges)  # Bulk add edges

# Compute Radiality Centrality efficiently
radiality_scores = radiality_centrality(G)

# Convert results to DataFrame
centrality_df = pd.DataFrame({
    "Author": radiality_scores.keys(),
    "Radiality_Centrality": radiality_scores.values(),
    "Class": [author_classes.get(node, "Nonawardee") for node in radiality_scores.keys()]
})

# Save results
centrality_df.to_csv("author_centralities_Radiality.csv", index=False)
print("Radiality Centrality calculated and saved.")


FileNotFoundError: [Errno 2] No such file or directory: '/content/combined_coauthors.csv'